In [2]:
import os
os.getcwd()
import pandas as pd

# Load full corpus
df = pd.read_csv("/Users/divyajyoti/dev/grad/NLP/ANLP-ESG/data/processed/intel_layout_all/corpus_pages_cleaned.csv")

df_amz = df.copy()

In [3]:
import re
import pandas as pd

def clean_page_text(text, patterns):
    """
    Remove all regex patterns from a given text.
    Patterns must be a list of regex strings.
    """
    if pd.isna(text):
        return text
    
    cleaned = text
    for pat in patterns:
        cleaned = re.sub(pat, " ", cleaned, flags=re.IGNORECASE | re.DOTALL)
    # collapse extra whitespace
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    return cleaned



In [4]:
import re
import pandas as pd

def final_super_clean(text):
    if pd.isna(text):
        return text

    # 1. Remove the classic PDF replacement character
    text = text.replace("�", " ")

    # 2. Remove the explicit weird control chars seen in reports
    text = text.replace("\x08", " ")   # BACKSPACE
    text = text.replace("\x05", " ")   # ENQUIRY

    # 3. Remove invisible Unicode characters (safe)
    text = re.sub(r"[\u200B-\u200D\uFEFF\u00AD]", " ", text)

    # 4. Remove ALL ASCII control characters (0x00–0x1F, 0x7F)
    #    This catches any remaining PDF junk
    text = re.sub(r"[\x00-\x1F\x7F]", " ", text)

    # 5. DO NOT remove non-ASCII (we keep em-dash, accents etc.)

    # 6. Collapse whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


In [5]:
df["page_text_final"] = df["page_text_final"].astype(str).apply(final_super_clean)

In [8]:
patterns = [

    # ======================================================
    # 1. Footer: "Corporate Responsibility Report www.intel.com/..."
    # ======================================================

    # Any year + "Corporate Responsibility Report www.intel.com/go/responsibility..."
    r"20\d{2}\s+Corporate\s+Responsibility\s+Report\s+www\.intel\.com/(?:go/)?responsibility(?:\s+\w+)*",

    # Generic footer (sometimes without leading year, with trailing section names)
    r"Corporate\s+Responsibility\s+Report\s+www\.intel\.com/(?:go/)?responsibility(?:\s+\w+)*",

    # Variant with just "/responsibility" (no /go/)
    r"Corporate\s+Responsibility\s+Report\s+www\.intel\.com/responsibility(?:\s+\w+)*",


    # ======================================================
    # 2. "Report Builder" header / navigation elements
    # ======================================================

    # "Access the Report Builder Corporate Responsibility Report www.intel.com/go/responsibility"
    r"Access\s+the\s+Report\s+Builder\s+Corporate\s+Responsibility\s+Report\s+www\.intel\.com/(?:go/)?responsibility",

    # More generic: "... Report Builder Corporate Responsibility Report www.intel.com/go/responsibility"
    r"Report\s+Builder\s+Corporate\s+Responsibility\s+Report\s+www\.intel\.com/(?:go/)?responsibility",


    # ======================================================
    # 3. "Table of Contents" navigation ribbons
    #    Example: "Table of Contents 2 Table of Contents Introduction 3 ... 92 Letter From Our CEO"
    # ======================================================

    # Any long "Table of Contents ..." ribbon, usually at the start of the page
    r"\bTable\s+of\s+Contents\b.*?(?=(Letter\s+From\s+Our\s+CEO|About\s+This\s+Report|Introduction|Customers|Company\s+Profile|Appendix|$))",

    # Simpler catch-all: standalone ToC lines
    r"^Table\s+of\s+Contents\b.*$",

    # Some pages start with a page number then ToC
    # e.g., "6 Table of Contents 2 Introduction 3 About This Report ..."
    r"^\s*\d+\s+Table\s+of\s+Contents\b.*$",


    # ======================================================
    # 4. "About This Report" header + nav blocks
    #    Example: "About this Report 4 About This Report Table of Contents 2 Introduction ..."
    # ======================================================

    r"\bAbout\s+This\s+Report\s+Table\s+of\s+Contents\b.*",

    # Standalone heading "About this Report" at top
    r"^About\s+This\s+Report\b.*",


    # ======================================================
    # 5. "For more information" boilerplate
    #    Example: "For more information, visit www.intel.com/go/responsibility"
    # ======================================================

    r"For\s+more\s+information,\s+visit\s+www\.intel\.com/(?:go/)?responsibility.*",

    # Variants like "... www.intel.com/go/responsibility" at end of page
    r"www\.intel\.com/(?:go/)?responsibility.*$",


    # ======================================================
    # 6. Letter-spaced section headings (obvious layout headings)
    #    Example: "C o r p o r a t e P r o f i l e a n d E c o n o m i c I m p ..."
    # ======================================================

    # "Corporate Profile and Economic Impact" rendered with spaces: C o r p o r a t e ...
    r"(?:C\s+o\s+r\s+p\s+o\s+r\s+a\s+t\s+e\s+P\s+r\s+o\s+f\s+i\s+l\s+e\s+a\s+n\s+d\s+E\s+c\s+o\s+n\s+o\s+m\s+i\s+c\s+I\s+m\s+p[^\s]*)",

    # Generic catch for spaced-out "Environmental Factors" / "Social Factors" style headings
    # (sequence of single letters separated by single spaces, then "Fa c t o r s")
    r"(?:E\s+n\s+v\s+i\s+r\s+o\s+n\s+m\s+e\s+n\s+t\s+a\s+l\s+Fa\s+c\s+t\s+o\s+r\s+s)",
    r"(?:S\s+o\s+c\s+i\s+a\s+l\s+Fa\s+c\s+t\s+o\s+r\s+s)",


    # ======================================================
    # 7. Generic Intel boilerplate at end of report pages
    # ======================================================

    # Trademarks / ownership line
    r"Other\s+names\s+and\s+brands\s+may\s+be\s+claimed\s+as\s+the\s+property\s+of\s+others\.$",

    # Common "total giving" line at end of some pages
    r"total\s+giving\s+\(cash\s+and\s+in-kind\)\s+from\s+Intel\s+Corporation\s+and\s+the\s+Intel\s+Foundation\.$",


    # ======================================================
    # 8. Leading page numbers (page numbers at line start)
    #    Example: " 6 Company Profile ...", "10 Customers ..."
    # ======================================================

    # Remove a leading integer + space at the very beginning (but keep the rest of the line)
    r"^\s*\d+\s+(?=\S)",
]


In [9]:
df_amz["page_text_final_clean"] = df_amz["page_text_final"].apply(
    lambda x: clean_page_text(x, patterns)
)


In [10]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 600)
pd.set_option('display.max_colwidth', None)
df_amz[["page", "page_text_final", "page_text_final_clean"]]

page  \
0        0   
1        1   
2        2   
3        3   
4        4   
...    ...   
1845   112   
1846   113   
1847   114   
1848   115   
1849   116   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             

In [11]:
df_amz.shape
df_amz = df_amz.dropna(
    subset=["page_text_final", "page_text_final_clean"]
).copy()
df_amz = df_amz[df_amz["page_text_final_clean"].notna()].copy()

df_amz.shape
df_amz["page_text_final"] = df_amz["page_text_final_clean"]
df_amz = df_amz.drop(columns=["page_text_final_clean"])
df_amz = df_amz.rename(columns={"page_text_final": "page_text"})
df_amz.to_csv(
    "/Users/divyajyoti/dev/grad/NLP/ANLP-ESG/data/processed/intel_layout_all/corpus_pages_clean_final.csv",
    index=False
)